# Module C live lab — mini-Lessmann on public credit data

**Paper:** Lessmann, Baesens, Seow & Thomas (2015), *EJOR* 247. Gunnarsson et al. (2021) is the XGBoost-era update.

**MODE = "live"** (default): GC and AC, 5×2 CV, fixed hyperparameters, raw `predict_proba`, `n_jobs=1`. Target: under 60 seconds. GMC is not fit here.

**MODE = "full"** does **not** fit here. The full protocol (inner 5-fold tuning + Platt) is `scripts/run_lessmann_full.py` (GC, AC) and `scripts/precompute_gmc.py` (GMC). Live mode loads those caches if present and prints the full rank table next to the live one.

**PREP_LINEAR = "robust"** (default): inside each training fold, clip each numeric at its 1st/99th percentile, log1p non-negative features with skew > 2, then StandardScaler — for LR, LR_L2 and MLP. `"standard"` is scaler only. Trees are unchanged.

GMC is a 30,000-row stratified subsample, precomputed (`data/cache/lessmann_gmc_{live,full}.pkl`).


## 0. Setup

In [ ]:
MODE = "live"  # "live" (classroom) or "full" (load cache only; do not fit the heavy protocol here)
PREP_LINEAR = "robust"  # "robust" (default) | "standard" — LR, LR_L2, MLP only; trees unchanged

from pathlib import Path
import sys
import time
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve
from sklearn.base import clone
from scipy.stats import beta as beta_dist, friedmanchisquare, norm

SEED = 2015
rng = np.random.default_rng(SEED)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True


def find_data_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        p = base / "data"
        if (p / "ff_daily_factors.csv").exists() or (p / "krauss_sp500_daily.csv").exists() or (p / "README.md").exists():
            return p.resolve()
    return Path.cwd() / "data"


def display_path(p):
    p = Path(p).resolve()
    for base in p.parents:
        if (base / "requirements.txt").is_file():
            try:
                return p.relative_to(base).as_posix()
            except ValueError:
                break
    return p.name


DATA = find_data_root()
CREDIT = DATA / "credit"
CACHE = DATA / "cache"
CREDIT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
SCRIPTS = DATA.parent / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

print("MODE", MODE)
print("PREP_LINEAR", PREP_LINEAR)
print("credit", display_path(CREDIT))
print("cache", display_path(CACHE))
HAS_GMC = any(CACHE.glob("lessmann_gmc_*.pkl"))
if not HAS_GMC:
    print("GMC skipped. To add it, place Kaggle Give Me Some Credit cs-training.csv in data/gmc/ and run: python scripts/precompute_gmc.py")


## 1. Data

GC and AC download from OpenML / UCI on the first run and are cached as `data/credit/gc.csv` and `data/credit/ac.csv`. After loading we print the target counts and assert Lessmann’s default rates (GC 0.300, AC 0.445, ±0.005). GMC (default rate 0.067 ± 0.005) is a 30,000-row stratified subsample, precomputed by `scripts/precompute_gmc.py`.


In [ ]:
import run_lessmann_full as L

L.PREP_LINEAR = PREP_LINEAR
print("PREP_LINEAR", L.PREP_LINEAR)

sets = L.load_gc_ac(CREDIT)
print("datasets:", list(sets))


## 2. Metrics (AUC, H-measure, partial Gini, Brier)

H-measure (Hand 2009, Beta(2,2)): from the ROC, \(L(c)=\min_t[\pi_0 c\,\mathrm{FPR}(t)+\pi_1(1-c)\,\mathrm{FNR}(t)]\), \(L_{\max}(c)=\min(\pi_0 c,\pi_1(1-c))\), \(H=1-\int L\,dF/\int L_{\max}\,dF\).

Partial Gini: Gini \(= 2\mathrm{AUC}-1\) on the subset with \(\hat p(+|x) \le 0.4\).


In [ ]:
h_measure = L.h_measure
partial_gini = L.partial_gini
all_metrics = L.all_metrics
make_preps = L.make_preps
mean_table = L.mean_table
rank_and_tests = L.rank_and_tests
expected_cost = L.expected_cost
cost_reductions = L.cost_reductions
print_cost_tables = L.print_cost_tables
reject_rates = L.reject_rates
print_reject_tables = L.print_reject_tables


def print_tests(res, label):
    print(f"\n{label} — Average metric ranks (1 = best)")
    print(res["avg_rank"].round(2))
    friedman_r = res["R"].sort_values().to_frame("R")
    print(f"\n{label} — Friedman ranks (R used in Friedman and Holm; 1 = best)")
    print(friedman_r.round(2))
    print(f"\nDatasets: {list(res['rank_mat'].index)}")
    print(f"Friedman chi2={res['friedman_chi2']:.3f}  p={res['friedman_p']:.4f}  on Friedman ranks across {res['N']} datasets.")
    print(f"N = {res['N']} datasets gives this test little power. That is why Lessmann used eight.")
    if res["friedman_p"] >= 0.05:
        print("\nExploratory: the Friedman test does not reject, so no pairwise claims (Demšar 2006; Lessmann follow the same rule)")
    print(f"\nLessmann Eq. (1) vs best={res['best']}: z=(R_i-R_best)/sqrt(k(k+1)/(6N)), k={res['k']}, N={res['N']}")
    print(f"  {res['best']:6s}  R={res['R_best']:.2f}  (best)")
    for m, Ri, z, pv, p_h in res["holm"]:
        print(f"  {m:6s}  R={Ri:.2f}  z={z:.3f}  p={pv:.3f}  p_Holm={p_h:.3f}  reject={p_h < 0.05}")


## 3. Live models (fixed budget, no inner CV, no Platt)

Linear models: one-hot + numeric prep from **PREP_LINEAR** (`robust` = clip 1st/99th, log1p if skew>2, then scale; `standard` = StandardScaler only). Trees stay unscaled (ordinal cats). `n_jobs=1`.

| model | spec |
|---|---|
| LR | unregularized logistic |
| LR_L2 | \(C=1\) |
| RF | 300 trees, `max_depth=8` |
| HGB | `max_depth=3`, `max_iter=200` |
| MLP | hidden `(16,)`, `solver=lbfgs`, `alpha=1.0`, `max_iter=2000` |
| AVG | simple mean of the five probability vectors |

`XGB` is sklearn `HistGradientBoosting`: the XGBoost/LightGBM wheels on this Mac need `libomp`.


In [ ]:
def live_models(lin, tree):
    return {
        "LR": Pipeline([("prep", lin), ("m", LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000))]),
        "LR_L2": Pipeline([("prep", lin), ("m", LogisticRegression(C=1.0, solver="lbfgs", max_iter=2000))]),
        "RF": Pipeline([("prep", tree), ("m", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=SEED, n_jobs=1))]),
        "HGB": Pipeline([("prep", tree), ("m", HistGradientBoostingClassifier(max_depth=3, max_iter=200, random_state=SEED))]),
        "MLP": Pipeline([("prep", lin), ("m", MLPClassifier(hidden_layer_sizes=(16,), solver="lbfgs", alpha=1.0, max_iter=2000, random_state=SEED))]),
    }


## 4. Repeated 5×2 CV (GC and AC). GMC loaded from cache.


In [ ]:
def run_live(name, X, y, n_repeats=5):
    cv = RepeatedStratifiedKFold(n_splits=2, n_repeats=n_repeats, random_state=SEED)
    rows = []
    oof_parts = []
    n_folds = n_repeats * 2
    for fold, (tr, te) in enumerate(cv.split(X, y)):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = np.asarray(y)[tr], np.asarray(y)[te]
        space = live_models(*make_preps(Xtr))
        probs = {}
        for mname, est in space.items():
            est = clone(est)
            est.fit(Xtr, ytr)
            p = est.predict_proba(Xte)[:, 1]
            probs[mname] = p
            rows.append({"dataset": name, "fold": fold, "model": mname, **all_metrics(yte, p)})
        avg = np.mean(np.vstack(list(probs.values())), axis=0)
        probs["AVG"] = avg
        rows.append({"dataset": name, "fold": fold, "model": "AVG", **all_metrics(yte, avg)})
        oof_parts.append(pd.DataFrame({"dataset": name, "fold": fold, "y": yte, **probs}))
        print(f"  {name} fold {fold+1}/{n_folds} done")
    return pd.DataFrame(rows), pd.concat(oof_parts, ignore_index=True)


live_tab = None
live_oof = None
live_secs = None
if MODE == "live":
    t0 = time.time()
    parts = []
    oof_parts = []
    for name in ["GC", "AC"]:
        X, y = sets[name]
        print(f"\n=== {name}  5×2 CV  LIVE ===")
        tab, oof = run_live(name, X, y, n_repeats=5)
        parts.append(tab)
        oof_parts.append(oof)
    live_tab = pd.concat(parts, ignore_index=True)
    live_oof = pd.concat(oof_parts, ignore_index=True)
    live_secs = time.time() - t0
    if HAS_GMC:
        gmc_live_path = L.gmc_pickle(CACHE, "live")
        if gmc_live_path.exists():
            gmc_live = pd.read_pickle(gmc_live_path)
            print("GMC is a 30,000-row stratified subsample, precomputed.")
            live_tab = pd.concat([live_tab, gmc_live["metrics"]], ignore_index=True)
            live_oof = pd.concat([live_oof, gmc_live["oof"]], ignore_index=True)
    print(f"\nlive CV runtime {live_secs:.1f}s")
    print(mean_table(live_tab))
else:
    print("MODE=full: not fitting in the notebook. Run scripts/run_lessmann_full.py and scripts/precompute_gmc.py")


(We discuss this result in class.)


In [ ]:
if HAS_GMC:
    rows = []
    for prep in ["standard", "robust"]:
        rec = {"prep": prep}
        for kind in ["live", "full"]:
            path = L.gmc_pickle(CACHE, kind, prep=prep)
            if not path.exists():
                continue
            pack = pd.read_pickle(path)
            tab = pack["metrics"] if isinstance(pack, dict) else pack
            auc = tab.groupby("model")["AUC"].mean()
            for m in ["LR", "LR_L2", "MLP"]:
                rec[(kind, m)] = float(auc[m])
        rows.append(rec)
    cmp = pd.DataFrame(rows).set_index("prep")
    cmp.columns = pd.MultiIndex.from_tuples(cmp.columns)
    print("GMC mean AUC (from caches, no refit)")
    print(cmp.round(4).to_string())


## 5. Average ranks, Friedman, Lessmann Eq. (1) + Holm

In [ ]:
full_tab = None
full_parts = []
for name in ["GC", "AC"]:
    p = L.full_pickle(CACHE, name)
    if p.exists():
        print("load full cache", p.name)
        full_parts.append(pd.read_pickle(p))
    else:
        print("full cache missing:", p.name, "— run scripts/run_lessmann_full.py")
if HAS_GMC:
    gmc_full_path = L.gmc_pickle(CACHE, "full")
    if gmc_full_path.exists():
        print("load full cache", gmc_full_path.name)
        full_parts.append(pd.read_pickle(gmc_full_path)["metrics"])
if full_parts:
    full_tab = pd.concat(full_parts, ignore_index=True)

if live_tab is not None:
    print_tests(rank_and_tests(live_tab), "LIVE")
if full_tab is not None:
    print("\nFULL mean metrics")
    print(mean_table(full_tab))
    print_tests(rank_and_tests(full_tab), "FULL protocol (precomputed)")
elif MODE == "full":
    raise SystemExit("Full cache not found. Run: python scripts/run_lessmann_full.py")


## 6. Cost curve (Lessmann Figure 2 style)

Fix \(C(+|-)=1\), vary \(C(-|+)\) from 2 to 50. Bayes threshold on **Platt-calibrated** out-of-fold probabilities from the full protocol: predict default if \(\hat p \ge 1/(1+C(-|+))\). Plot and tables: % cut in expected cost vs LR, then the share of applicants rejected — GC, AC, GMC and the three-dataset mean at \(C=2,5,10,25,50\).


In [ ]:
def print_cost_and_reject(oof, label):
    by_ds, overall = cost_reductions(oof)
    print(label)
    print_cost_tables(by_ds, overall)
    rej_ds, rej_ov = reject_rates(oof)
    print()
    print_reject_tables(rej_ds, rej_ov)
    return by_ds, overall


oof_parts = []
oof_path = L.full_oof_pickle(CACHE)
if oof_path.exists():
    oof_parts.append(pd.read_pickle(oof_path))
else:
    print("OOF cache missing:", oof_path.name, "— run scripts/run_lessmann_full.py")
if HAS_GMC:
    gmc_full_path = L.gmc_pickle(CACHE, "full")
    if gmc_full_path.exists():
        oof_parts.append(pd.read_pickle(gmc_full_path)["oof"])
if oof_parts:
    oof = pd.concat(oof_parts, ignore_index=True)
    by_ds, overall = print_cost_and_reject(oof, "FULL (Platt-calibrated)")
    fig, ax = plt.subplots()
    for m, g in overall.groupby("model"):
        if m == "LR":
            continue
        ax.plot(g["C"], g["red"], label=m)
    ax.axhline(0, color="0.5", lw=1)
    ax.set_xlabel(r"$C(-|+)$  (with $C(+|-)=1$)")
    ax.set_ylabel("% cut in expected cost vs LR")
    ax.set_title("OOF calibrated CV — mean over GC, AC and GMC")
    ax.legend(ncols=2, fontsize=8)
    plt.tight_layout()
    plt.show()


Uncalibrated probabilities: for contrast only (Bayes thresholds need calibration)


In [ ]:
if live_oof is not None:
    print_cost_and_reject(live_oof, "LIVE (uncalibrated)")
else:
    print("No live OOF (MODE=full). Uncalibrated contrast is available in live mode.")


## Takeaway

(We discuss this result in class.)

Live mode is the classroom sketch. GC/AC full numbers live in `data/cache/lessmann_full_*.pkl`; GMC is a 30,000-row stratified subsample, precomputed.

**Homework (optional):** `03_invoice_timing_simplified.ipynb` — random vs chronological vs customer-grouped split.
